In [1]:
import pandas as pd
import pulp
import numpy as np

In [2]:
# 1. Load the Datasets (Your Excel/CSV files)
df_items = pd.read_csv("mto_item_master.csv")
df_machines = pd.read_csv("machine_capacities.csv")
df_routings = pd.read_csv("production_routings.csv")
df_quotes = pd.read_csv("external_supplier_quotes.csv")
df_orders = pd.read_csv("active_indents.csv")

In [3]:
# 2. Pre-Processing & Merging Data
# Merge all relevant data onto the active orders to create a flat analytical view
df = df_orders.merge(df_items, on="item_id").merge(df_quotes, on="item_id")

In [4]:
# Calculate Net Machine Capacity (Available - WIP)
machine_net_cap = {}
machine_rates = {}
for _, row in df_machines.iterrows():
    m_id = row['machine_id']
    machine_net_cap[m_id] = max(0, row['available_weekly_hours'] - row['existing_wip_hours'])
    machine_rates[m_id] = row['operating_cost_per_hour']

In [5]:
# Calculate True Cost to Buy
df['true_buy_cost'] = df['vendor_price_per_unit'] + df['incoming_qa_cost_per_unit']

In [6]:
# 3. Initialize the PuLP MILP Model
model = pulp.LpProblem("IOL_Make_vs_Buy_Optimization", pulp.LpMinimize)

In [7]:
# Decision Variables: How many units of each order to Make vs. Buy
# Indexed by order_id
make_vars = model.add_variable_dicts("Make", df['order_id'], lowBound=0, cat='Integer')

In [8]:
buy_vars = model.add_variable_dicts("Buy", df['order_id'], lowBound=0, cat='Integer')

In [9]:
# 4. Objective Function: Minimize Total Cost
objective_terms = []
for _, row in df.iterrows():
    order = row['order_id']
    # Cost to make = Raw Material (factoring scrap)
    rm_cost = row['raw_material_cost_per_unit'] * (1 + row['expected_scrap_percentage'])
    # Machine operating costs will be added implicitly via routing hours, but let's add RM now
    objective_terms.append(make_vars[order] * rm_cost)
    # Cost to buy
    objective_terms.append(buy_vars[order] * row['true_buy_cost'])

In [10]:
model += pulp.lpSum(objective_terms)

In [11]:
# 5. Constraints
for _, row in df.iterrows():
    order = row['order_id']
    
    # Constraint 1: Fulfill total demand for the order
    model += make_vars[order] + buy_vars[order] == row['order_quantity'], f"Demand_{order}"
    
    # Constraint 2: Strategic Override (Critical items MUST be made in-house)
    if row['is_critical_component']:
        model += buy_vars[order] == 0, f"Strategic_Make_{order}"

In [12]:
# Ensure the total machine hours required for all "Make" decisions do not exceed Net Capacity
for m_id in machine_net_cap.keys():
    machine_usage = []
    for _, row in df.iterrows():
        order = row['order_id']
        item = row['item_id']
        scrap_factor = 1 + row['expected_scrap_percentage']
        
        # Find routing hours for this specific item on this specific machine
        routing_step = df_routings[(df_routings['item_id'] == item) & (df_routings['machine_id'] == m_id)]
        if not routing_step.empty:
            # Add setup time (simplified here as per-unit distribution for linear scaling, 
            # or treat as a flat deduction if using binary variables)
            run_hrs = routing_step['run_hours_per_unit'].values[0]
            
            # Machine usage = Qty * Run Hours * Scrap Factor
            machine_usage.append(make_vars[order] * run_hrs * scrap_factor)
            
            # Add machine operating cost to objective function
            model.objective += make_vars[order] * (run_hrs * scrap_factor * machine_rates[m_id])
            
    if machine_usage:
        model += pulp.lpSum(machine_usage) <= machine_net_cap[m_id], f"CapLimit_{m_id}"

In [13]:
# 6. Solve the Model
stats = model.solve()

CBC devel (git:146ce89) — COIN-OR Branch and Cut
  args: /tmp/9605c0ff8aaf44ba896fd062063df52f-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /tmp/9605c0ff8aaf44ba896fd062063df52f-pulp.sol
  default strategy 1

▶ Problem loading

  Problem: MODEL
  |Rows| = 138   |Cols| = 200   |NZ| = 403
  Variables: 0 binary, 200 integer, 0 continuous

  Coefficient ranges:
    Matrix  ∈ [0.13286, 2.6478]   κ ≈  19.93
    Cost    ∈ [ 1093.9,  60582]   κ ≈  55.38
    Bounds  ∈ [      -,      -]   κ ≈      -
    RHS     ∈ [     11,   2340]   κ ≈ 212.73

✔ Problem loaded — 138 rows, 200 cols, 403 NZ, 200 integer

▶ Root LP relaxation

  Bound tightening: 200 bounds tightened (13 fixed) in 0.000 s.
  Clique strengthening: no changes (0.00s)
  Coefficient strengthening: no changes (0.00s)
  Row reductions: 13 rows removed (13 all-fixed, 0 duplicate, 0 parallel), 0 row bounds tightened (0.00s)
  LP auto: recommended dual_pesteep_psi1_pertv61
✔ LP Optimal — Frac: 0/200 (0.0%)   Obj: 1.3558

In [14]:
results = []

In [15]:
for _, row in df.iterrows():
    order = row['order_id']
    results.append({
        'Order_ID': order,
        'Item_ID': row['item_id'],
        'Demand': row['order_quantity'],
        'Make_In_House_Qty': make_vars[order].varValue,
        'Buy_External_Qty': buy_vars[order].varValue,
        'Is_Critical': row['is_critical_component'],
    })

In [16]:
df_results = pd.DataFrame(results)
df_results.to_csv("Final_Make_Buy_Decisions.csv", index=False)

In [17]:
print("Optimization Complete. Status:", stats.status.name)

print("Results saved to Final_Make_Buy_Decisions.csv")

Optimization Complete. Status: Optimal
Results saved to Final_Make_Buy_Decisions.csv
